# Qwen2.5-0.5B Fine-tuning Pipeline - Medical Chatbot

This notebook fine-tunes Qwen2.5-0.5B-Instruct on CPU using LoRA with MedDialog dataset.

In [1]:
import os
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, load_dataset
from trl import SFTTrainer
from tqdm import tqdm

In [2]:
print("=" * 50)
print("Environment Setup")
print("=" * 50)
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Environment Setup
PyTorch version: 2.11.0
CUDA available: False
Device: cpu


## 1. Load and Prepare Dataset

In [3]:
DATASET_SIZE = 500  # Reduced for CPU training
DATA_PATH = "OpenMed/MedDialog"
OUTPUT_DIR = "./medchat_output"

print("Loading dataset from Hugging Face...")
ds = load_dataset(DATA_PATH)
df = ds["train"].to_pandas()
print(f"Original dataset size: {len(df)}")

df = df.sample(n=min(DATASET_SIZE, len(df)), random_state=42)
print(f"Using dataset size: {len(df)}")
df.head(2)

Loading dataset from Hugging Face...


Original dataset size: 226557
Using dataset size: 500


,patient_message,doctor_response,dialogue_context
202094,What causes cough and shortness of breath?\n\n...,Thanks for your question on Healthcare Magic.I...,
156318,Q. Watery semen leaks out automatically. Is it...,Hello. I would like to tell you that masturbat...,


## 2. Format Data for Training

In [4]:
def format_conversation(example):
    patient_msg = str(example.get("patient_message", ""))
    doctor_response = str(example.get("doctor_response", ""))
    return {
        "text": f"Patient: {patient_msg.strip()}\n\nDoctor: {doctor_response.strip()}<|endoftext|>"
    }

formatted_data = df.apply(format_conversation, axis=1).tolist()
dataset = Dataset.from_list(formatted_data)
print(f"Formatted {len(dataset)} samples")

Formatted 500 samples


## 3. Load Model and Tokenizer

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="cpu",
    trust_remote_code=True
)
model.config.use_cache = False
print(f"Model loaded. Parameters: {model.num_parameters():,}")

Loading tokenizer...
Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded. Parameters: 494,032,768


## 4. Configure LoRA

In [6]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## 5. Training Configuration

In [7]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    fp16=False,
    logging_steps=50,
    save_steps=100,
    save_total_limit=2,
    warmup_steps=10,
    report_to="none",
    remove_unused_columns=False,
)

print("Training arguments configured")

Training arguments configured


## 6. Initialize Trainer

In [8]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=lambda example: example["text"],
)
print("Trainer initialized")

Applying formatting function to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer initialized


## 7. Start Training

In [9]:
print("=" * 50)
print("Starting training...")
print("=" * 50)
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Starting training...


/Users/nnhoang/nnhoang/tdtu/NLP/final/Q3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 

## 8. Save Model

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Model saved to ./medchat_output


## 9. Test Inference

In [ ]:
test_question = "I have a severe headache for 3 days. What should I do?"
print(f"Test Question: {test_question}\n")

messages = [
    {"role": "system", "content": "You are a medical assistant. Provide helpful and accurate health information."},
    {"role": "user", "content": test_question}
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
device = next(model.parameters()).device
inputs = tokenizer([text], return_tensors="pt").to(device)

model.eval()
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Answer: {response}")

Test Question: I have a severe headache for 3 days. What should I do?

Answer: patient:\nI have a severe headache for 3 days. What should I do?
doctor:\n\n\n\nDear, Thanks for writing to us. I understand your concern about your headache.
The information provided is limited, so it's difficult to accurately diagnose the cause. Usually, severe headache could be due to migraine, tension-type headache, cluster headache, or some other serious underlying condition, like brain hemorrhage, meningitis, or brain tumor. But don't worry too much. I would like to know more details to help better: _\n\n\n\n\n1. Is the headache localized (one side or both sides)?\n\n\n\n2. Is it throbbing (like beating) or压迫性 (pressure-like)?\n\n\n\n3. Associated symptoms like nausea, vomiting, fever, vision changes, or weakness?\n\n\n\nIf you have any of these associated symptoms, please seek immediate medical care. Also, if the headache is getting worse or not relieved by painkillers, please see a doctor for further

In [ ]:
print("=" * 50)
print("Training complete!")
print("=" * 50)

Training complete!
